# BBM ↔ external-platform harmonization — findings

Extends Kholmatova (2026), *Breakdowns in Realizing the Digital Extended Specimen*, from the paper's 131-record sample to the full BBM fungal collection, and adds automated cross-platform record resolution.

**Harmonization relationships** (README): for a BBM specimen and its counterpart on a public platform —
- **bidirectional** — we cite their id *and* they cite our catalog number back
- **unidirectional** — only one side cites the other (either direction)
- **absent** — same specimen, cited nowhere in either direction

Every number below is produced live by calling the pipeline scripts. Network-dependent cells are marked; run the fetch scripts first (`get_bbm_records.py`, `get_mo_records.py`).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "scripts" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))
import link_audit as la
print("repo:", ROOT)

## 1. BBM → Mushroom Observer (lookup by stored id)

MO is an **independent, upstream** platform — the Ceskas posted there directly, so MO holds no copy of our GUID. The only link is the id we recorded (`MO # 82752`). We scan `bbm_records.csv` for those, look each up on MO, and check whether MO cites us back (a free-text `UBC F#` note).

In [ ]:
# NETWORK: queries Mushroom Observer
mo = la.MushroomObserver()
res = la.audit(mo)                     # scan -> lookup -> classify
c = res["counts"]
on_mo = c["bidirectional"] + c["unidirectional"]
print(f"BBM records scanned     : {res['n_rows']}")
print(f"records citing an MO id  : {res['n_with_ref']}  ({100*res['n_with_ref']/res['n_rows']:.2f}%)")
print(f"distinct MO ids          : {len(res['ref_map'])}")
print(f"  resolve on MO          : {on_mo}")
print(f"    bidirectional        : {c['bidirectional']}")
print(f"    unidirectional UBC→MO: {c['unidirectional']}")
print(f"  dangling               : {c['dangling']}")

**Result (last full run):** of **34,856** records only **20** cite an MO id (**0.06 %**), across **19** distinct ids — **17 bidirectional, 2 unidirectional (UBC→MO), 0 dangling.**

The sparseness *is* the paper's finding (breakdown category 01, *missing identifier cross-references*), now measured collection-wide instead of in an 80-record sample. The high bidirectional share among the few that are linked reflects (a) we only see records that already cite MO, and (b) MO now carries `Herbarium Specimen: UBC F#` notes added by later harmonization work.

## 2. MyCoPortal (harvested — matched by GUID)

MyCoPortal is the **opposite coupling**: it is **harvested wholesale from our Specify database** via Symbiota, so every MP record carries our GUID (`occurrenceID`) and F-number (`catalogNumber`) *by propagation*. The `Mycoportal # UBC#####` strings in our records are **legacy free-text, not queryable ids** (they even ride along verbatim into MP's `occurrenceRemarks`). The reliable link is the **GUID**.

Confirmed against the live Symbiota API (`/api/v2/occurrence?occurrenceID=<guid>`):
- UBC fungi on MyCoPortal (`collid 49`): **34,946 records** ≈ our 34,856 — essentially the whole collection.
- match: MP `occurrenceID` == BBM `guid`; MP `catalogNumber` == BBM `F#`.

**Coupling contrast, quantified:** loosely-coupled MO → **0.06 %** cross-referenced; tightly-coupled MyCoPortal → **~complete**. That is the paper's central argument in two numbers. (A GUID-discovery audit that counts MP presence + harvest gaps per record is the natural next script.)

In [ ]:
# OFFLINE: how many records carry the legacy 'Mycoportal #' annotation
mp = la.MyCoPortal()
ref_map, n_rows, n_with_ref = la.scan(mp, str(la.INPUT))
print(f"records with a legacy 'Mycoportal #' note : {n_with_ref} (of {n_rows})")
print("→ not a queryable id; real MP linkage is the GUID (see above)")

## 3. Ceska / Observatory Hill quadrants — cross-platform resolution (README §2)

To answer *how many MO records are ours but unconnected* we resolve records **by attributes**, using the evaluator subsystem **vendored** into `scripts/evaluators/` (copied from the orchestration framework) — `RuleBasedEvaluator` + `LLMEvaluator` — via `resolve.py`. Both platforms are shaped into orchestration's record contract, blocked by genus, and clustered; a cluster with a BBM and an MO record is a match. Each match is then scored into a quadrant by crossing the attribute match with the recorded cross-references.

**Seeds:** MO user 2873 (Ceska) = 5,866 obs; MO location 1679 (Observatory Hill) = 2,707 obs.

> Requires `bbm_records.csv` (joined) and `mo_records.csv`. Set `LLM_MODEL` in `.env` to enable the LLM tier.

In [ ]:
import resolve as R
from collections import Counter
bbm_p, mo_p = R.DATA_DIR / "bbm_records.csv", R.DATA_DIR / "mo_records.csv"

if bbm_p.exists() and mo_p.exists():
    bbm_rows, bmeta = R.load_bbm(str(bbm_p))
    mo_rows, mmeta = R.load_mo(str(mo_p))
    meta = {**bmeta, **mmeta}
    pairs = R.resolve(bbm_rows, mo_rows, meta, use_llm=True)   # NETWORK if LLM_MODEL set
    q = Counter(R.quadrant(b, m, meta) for b, m, _ in pairs)
    how = Counter(h for _, _, h in pairs)
    print(f"BBM records            : {len(bbm_rows)}")
    print(f"MO Ceska/OH records    : {len(mo_rows)}")
    print(f"cross-platform matches : {len(pairs)}   (by method: {dict(how)})")
    for k in ("bidirectional","unidirectional_UBC_to_MO","unidirectional_MO_to_UBC","absent"):
        print(f"  {k:28} {q.get(k,0)}")
else:
    print("Run get_bbm_records.py and get_mo_records.py first, then re-run this cell.")

**How to read it:** `absent` is the payoff of resolution — MO records that are the same specimen as a BBM holding but linked *nowhere*. The **estimate** the README asks for (how many MO records likely correspond to BBM holdings) is `matches` + the confident `absent`/`MO→UBC` counts; whether Ceska-OR-Observatory-Hill is the right net is judged by how many MO records match no BBM record at all.

## 4. Comparison to Kholmatova (2026), Figure 2

Figure 2 (Phase I) traced **80 MO** records to UBC: **34 matched** → 8 bidirectional / 18 unidirectional / 8 absent; **19** of those UBC records carried an MO reference (the like-for-like slice for our lookup audit).

In [ ]:
import pandas as pd
paper = {"UBC records citing MO":19, "bidirectional":8,
         "unidirectional (UBC→MO)":10, "dangling / wrong id":1, "bidirectional rate":"42%"}
ours  = {"UBC records citing MO":len(res["ref_map"]), "bidirectional":c["bidirectional"],
         "unidirectional (UBC→MO)":c["unidirectional"], "dangling / wrong id":c["dangling"],
         "bidirectional rate":f"{100*c['bidirectional']/max(len(res['ref_map']),1):.0f}%"}
pd.DataFrame({"Kholmatova 2026 (Fig 2, Phase I)":paper, "This audit (full collection)":ours})

We **reproduce and extend** the paper's core claim (cross-references are almost always missing) and cannot reproduce Figure 2's full quality split from the lookup direction alone — that needs the two-directional, attribute-matched resolution in §3, which surfaces the MO→UBC and absent quadrants the paper found.

## Methods & caveats

- **BBM data**: full `collectionobject` table + joins to determination→taxon, collector→agent, collecting-event→locality (`get_bbm_records.py`).
- **MO reference formats caught**: `MO # 82752`, `MUOB 12345`, `Mushroom Observer observation #…`, mushroomobserver.org URLs. `MO posted as …` (no number) is a link with no id and is not looked up.
- **Coupling dictates method**: harvested-downstream platforms (MyCoPortal, GBIF) match by our GUID; independent platforms (MO, GenBank) match by the id we stored + attribute resolution.
- **Resolution**: rule-based predicates use name + exact date + locality/collector *token overlap* (cross-platform strings are formatted differently); the LLM tier adjudicates the ambiguous middle when configured.
- **Not yet built**: MyCoPortal GUID-discovery audit; GBIF / GenBank (both need the discovery direction).